# Local Liquid d1 — RAGTruth unsteered vs steered

Use a Colab GPU runtime. This contains **no paid API calls**. The `system_one` interface is documented; activation hook compatibility is unverified and stops if it fails.

In [ ]:
from google.colab import files
import pathlib,zipfile,os,subprocess
uploaded=files.upload()
packet=next((x for x in uploaded if x.startswith("FEMI_NEXT_EXPERIMENTS_") and x.endswith(".zip")),None)
assert packet
with zipfile.ZipFile(packet) as z:z.extractall("/content")
root=pathlib.Path("/content/FEMI_NEXT_EXPERIMENTS_20261108")
os.chdir(root)

## 1 — Offline tests and dataset acquisition
RAGTruth is hosted at https://github.com/ParticleMedia/RAGTruth. The following `git clone` obtains current public dataset files; record the commit hash for provenance.

In [ ]:
subprocess.run(["python","-m","unittest","discover","-s","tests","-q"],check=True)
subprocess.run(["git","clone","--depth","1","https://github.com/ParticleMedia/RAGTruth.git","/content/RAGTruth"],check=True)
subprocess.run(["git","-C","/content/RAGTruth","rev-parse","HEAD"],check=True)

## 2 — Freeze source-disjoint RAGTruth rows

In [ ]:
subprocess.run(["python","src/prepare_d1.py","--response-file","/content/RAGTruth/dataset/response.jsonl","--source-file","/content/RAGTruth/dataset/source_info.jsonl","--out","runs/d1","--train-pairs","24","--dev","40","--test","100"],check=True)

## 3 — GPU setup and hook inspection
Select GPU runtime with sufficient memory. The model loads remote code: inspect its model card and grant access only if trusted.

In [ ]:
subprocess.run(["pip","install","-r","requirements-gpu.txt"],check=True)
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
print(torch.cuda.get_device_name())

In [ ]:
# This loads d1 and lists potential layer modules.
subprocess.run(["python","src/d1_local.py","list-hooks","--dir","runs/d1"],check=True)

## 4 — Fit direction on source-matched training pairs; tune dose on dev only
Identify an actual once-per-call `[batch, seq, hidden]` module from the previous output. An incompatible hook stops execution. Do NOT modify the study based on test outcomes.

In [ ]:
HOOK_NAME="REPLACE_WITH_EXACT_MODULE_NAME"
assert not HOOK_NAME.startswith("REPLACE"),"Choose a verified module path from list-hooks"
# subprocess.run(["python","src/d1_local.py","fit","--dir","runs/d1","--hook-name",HOOK_NAME],check=True)

## 5 — One-shot heldout test, four arms
After fitting, invoke once. The outputs include unsteered, instruction-steered, activation-steered and random sham; no synthetic test outcomes are included in this packet.

In [ ]:
# subprocess.run(["python","src/d1_local.py","test","--dir","runs/d1","--hook-name",HOOK_NAME],check=True)